In [ ]:
# main.py
import sys
from typing import cast

from PyQt5.QtWidgets import QApplication, QMainWindow, QWidget

from card_panel import CardPanel
from game import Game
from ui_mainwindow import Ui_MainWindow


class MainWindow(QMainWindow):
    def __init__(self) -> None:
        super().__init__()

        # Strongly typed UI object
        self.ui: Ui_MainWindow = Ui_MainWindow()
        self.ui.setupUi(self)

        # Cast cpHand1 to CardPanel so Pylance knows its real type
        self.hand_panel: CardPanel = cast(CardPanel, self.ui.cpHand1)

        # Connect click signal
        self.hand_panel.card_clicked.connect(self.on_card_clicked)

        # Create game controller
        self.game: Game = Game(sequences=2)

        # Start a new game immediately
        self.start_new_game()

    # ---------------------------------------------------------
    # GAME INTEGRATION
    # ---------------------------------------------------------
    def start_new_game(self) -> None:
        """Create a new game and update the GUI."""
        self.game.create_new_game()

        # Later: update card panel with actual card images
        # self.hand_panel.update_cards(self.game.hand1)
        # self.hand_panel.set_cards(
        print("need to implement card panel update with game hand1")

    def on_card_clicked(self, index: int) -> None:
        card = self.game.get_card(index)
        print(f"Clicked card index: {index}, card: {card}")


def main() -> None:
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()


In [ ]:
# main.py
import sys
from typing import cast

from PyQt5.QtWidgets import QApplication, QMainWindow, QWidget

from card_panel import CardPanel
from game import Game
from ui_mainwindow import Ui_MainWindow


class MainWindow(QMainWindow):
    def __init__(self) -> None:
        super().__init__()

        # Strongly typed UI object
        self.ui: Ui_MainWindow = Ui_MainWindow()
        self.ui.setupUi(self)

        # Cast cpHand1 to CardPanel so Pylance knows its real type
        self.hand_panel: CardPanel = cast(CardPanel, self.ui.cpHand1)

        # Connect click signal
        self.hand_panel.card_clicked.connect(self.on_card_clicked)

        # Create game controller
        self.game: Game = Game(sequences=2)

        # Start a new game immediately
        self.start_new_game()

    # ---------------------------------------------------------
    # GAME INTEGRATION
    # ---------------------------------------------------------
    def start_new_game(self) -> None:
        """Create a new game and update the GUI."""
        self.game.create_new_game()

        # Later: update card panel with actual card images
        # self.hand_panel.update_cards(self.game.hand1)
        # self.hand_panel.set_cards(
        print("need to implement card panel update with game hand1")

    def on_card_clicked(self, index: int) -> None:
        card = self.game.get_card(index)
        print(f"Clicked card index: {index}, card: {card}")


def main() -> None:
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()


In [ ]:
# card.py
from typing import Optional


class Card:
    """
    Simple data container for a Pinochle card.
    """

    def __init__(
        self,
        suit: str,
        rank: str,
        filename: str,
        copy: int = 1,
        index: Optional[int] = None,
    ):
        self.suit = suit
        self.rank = rank
        self.filename = filename
        self.copy = copy  # 1..N (sequence number)
        self.index = index  # global deck index
        print(f"Card __init__, self.copy = {self.copy}, self.index = {self.index}")

    def __repr__(self):
        return f"{self.rank} of {self.suit} (copy {self.copy})"


In [ ]:
# deck.py
import os
import random
from typing import Dict, List

from card import Card

SUITS = ["hearts", "spades", "diamonds", "clubs"]
RANKS = ["A", "10", "K", "Q", "J", "9"]


class Deck:
    """
    Deck builder for Pinochle-style decks.
    Supports:
        - 1 sequence  (24 cards)  → "half deck"
        - 2 sequences (48 cards)  → standard pinochle deck
        - 4 sequences (96 cards)  → double pinochle deck

    Cards of identical suit+rank are kept adjacent.
    """

    def __init__(self, image_dir: str = "images\\cards", sequences: int = 2):
        self.image_dir = image_dir
        self.sequences = max(1, int(sequences))  # 1, 2, or 4
        self.cards: dict[int, Card] = {}  # index → Card
        self.indices: List[int] = []  # list of indices 1..N

    # ---------------------------------------------------------
    # BUILD THE DECK
    # ---------------------------------------------------------
    def build_deck(self) -> Dict[int, Card]:
        """
        Build the deck dictionary with adjacent duplicates.
        Index order example for sequences=2:
            hearts_A_1
            hearts_A_2
            hearts_10_1
            hearts_10_2
            ...
        """
        self.cards.clear()
        self.indices.clear()

        index: int = 1

        for suit in SUITS:
            for rank in RANKS:

                # Build the image filename
                filename = os.path.join(self.image_dir, f"{suit}_{rank}.png")

                # Add N copies of this card (adjacent)
                for copy_num in range(self.sequences):
                    self.cards[index] = Card(
                        suit=suit,
                        rank=rank,
                        filename=filename,
                        copy=copy_num + 1,  # 1..N
                        index=index,
                    )
                    self.indices.append(index)
                    index += 1

        return self.cards

    # ---------------------------------------------------------
    # SHUFFLE
    # ---------------------------------------------------------
    def shuffle(self) -> List[int]:
        """Shuffle the list of card indices."""
        random.shuffle(self.indices)
        return self.indices

    # ---------------------------------------------------------
    # DEAL
    # ---------------------------------------------------------
    def deal(self, n: int) -> List[int]:
        """
        Deal n cards by index.
        Removes dealt cards from the deck.
        Note: this modifies the deck's indices list.
        Not what I expected when I wrote the method name,
        need to understand better but it works for now.
        """
        hand: List[int] = self.indices[:n]
        del self.indices[:n]
        return hand


In [ ]:
# card_panel.py
"""
CardPanel class is for displaying a row of card images.

CardPanel is a widget that displays a list of card images in a horizontal layout.
Each card is represented by a QLabel containing an image.

The class provides methods to set the cards to be displayed and to clear the panel.

This class emits a signal when a card is clicked, allowing other parts of the
application to respond to user interactions.

The card images are expected to be located in a "cards" directory, with filenames
corresponding to their card IDs (e.g., "23.png", "24.png", etc.).

The card dimensions are defined by the constants CARD_W and CARD_H.

This class is designed to be used in a PyQt5 application and can be integrated
into a larger GUI layout.

Typical usage involves creating an instance of CardPanel, setting the cards to display,
and connecting the card_clicked signal to a slot that handles the click events.

The class is flexible and can be extended with additional functionality, such as
support for different card sizes, additional signals for other interactions, or
methods for updating the card images dynamically.

This class was created by Copilot and may require further customization to fit
specific application needs.

This class is an extension of QWidget and was created by promoting the PyQT5 QWidget class in Qt Designer. It is designed to be used as a custom widget in a PyQt5 application.
QTextBrowser by instructions from Copilot as follows:

    How this connects to your Designer widget (cpHand1)
    You changed your container to a QTextBrowser named cpHand1.

    That’s fine — because you will promote it to CardPanel.

    Steps:
    In Qt Designer, select cpHand1

    Right‑click → Promote to…

    Base class: QWidget

    Promoted class name: CardPanel

    Header file: card_panel

    Click Add, then Promote

    Now your QTextBrowser becomes a CardPanel at runtime.

Copilot generated the CardPanel class code based on the
overall project description and requirements provided. I created
the module card_panel.py and copied and pasted the code from Copilot
responses. Copilot was a separate entity that operated in a Edge browser
outside the VS Code application. This was not the GitHub Copilot.

It is interesting to note that the mainwindow.ui XML filespecifies that
CardPanel extends QTextBrowser and a <header>card_panel.py</header>. This header
file name was set in Qt Designer as part of the promotion process.

The follow snippet is the relevant portion of the mainwindow.ui XML
file that shows the promotion:

 <customwidgets>
  <customwidget>
   <class>CardPanel</class>
   <extends>QTextBrowser</extends>
   <header>card_panel.py</header>
  </customwidget>
 </customwidgets>

 The ui_mainwindow.py file generated by pyuic5 includes the import of CardPanel class
 from card_panel.py module as the last line of the file.
 This allows the MainWindow class to create an instance of CardPanel
 and use it as a widget in the GUI.

 from card_panel import CardPanel


Attributes:
    attribute1 (type): Description of attribute1.
    attribute2 (type): Description of attribute2.

Methods:
    method_name: Optional section. Usually omitted because methods
        have their own docstrings.

Examples:
    Example usage can go here:

    >>> obj = ClassName(value)
    >>> obj.do_something()
"""

from __future__ import annotations

from typing import List

from PyQt5.QtCore import Qt, pyqtSignal
from PyQt5.QtGui import QPixmap
from PyQt5.QtWidgets import QHBoxLayout, QLabel, QWidget

CARD_W: int = 100
CARD_H: int = 150

import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
CARD_DIR = os.path.join(BASE_DIR, "images", "cards")


class CardPanel(QWidget):
    """
    A horizontal panel of clickable card images.
    Emits card_clicked(index) when a card is clicked.
    """

    card_clicked = pyqtSignal(int)

    def __init__(self, parent: QWidget | None = None) -> None:
        super().__init__(parent)

        # Internal state
        self.cards: List[int] = []  # card IDs (e.g., 23, 24, 22)
        self.labels: List[QLabel] = []  # QLabel widgets for each card

        # Layout
        self.layout: QHBoxLayout = QHBoxLayout()
        self.layout.setSpacing(8)
        self.layout.setContentsMargins(0, 0, 0, 0)
        self.setLayout(self.layout)

    # ---------------------------------------------------------
    # Load card images into the panel
    # ---------------------------------------------------------


def set_cards(self, card_list: list[int]) -> None:
    """Replace all cards with a new list of card IDs."""
    self.clear()
    self.cards = card_list

    for index, card_id in enumerate(card_list):
        card = self.parent().game.get_card(card_id)  # get Card object

        suit = card.suit.lower()  # clubs, hearts, etc.
        rank = card.rank  # A, 10, K, Q, J, 9

        filename = f"{suit}_{rank}.png"
        print("filename in set cards", filename)# DEBUG
        full_path = os.path.join(CARD_DIR, filename)
        print("Loading:", full_path)  # DEBUG

        lbl = QLabel(self)
        lbl.setFixedSize(CARD_W, CARD_H)
        lbl.setScaledContents(True)

        pix = QPixmap(full_path)
        lbl.setPixmap(pix)

        lbl.mousePressEvent = lambda e, i=index: self.card_clicked.emit(i)

        self.labels.append(lbl)
        self.layout.addWidget(lbl)

    # ---------------------------------------------------------
    # Remove all cards
    # ---------------------------------------------------------
    def clear(self) -> None:
        """Remove all card widgets from the panel."""
        for lbl in self.labels:
            self.layout.removeWidget(lbl)
            lbl.deleteLater()

        self.labels = []
        self.cards = []
